# FMSE Lab 13 - Break the Agent

**Course:** FMSE 2026 · **Module 13:** AI Security, Red Teaming, and Safety Architecture · **Phase:** Verify & Validate · **Tier:** Senior

Course home and progress: [agentic-ai.es/academy/fmse/learn/m13-security-red-teaming](https://agentic-ai.es/academy/fmse/learn/m13-security-red-teaming).

## 0. Mission and competency target

**Mission.** Discover at least four planted vulnerabilities, implement architectural mitigations, and prove the mitigations with regression tests.

**Guided lab.** Students attack a deliberately vulnerable agent and inspect traces.

**Competency target (Module 13 outcomes):**
- Threat-model direct/indirect prompt injection, sensitive data disclosure, supply-chain/context poisoning, improper output handling, excessive agency, and unbounded consumption.
- Implement least privilege and approval gates outside the model.
- Red-team retrieval/tool systems and convert successful attacks into regression tests.
- Design incident logging, kill switches, and safe degraded modes.

**Scaffolding:** about 25% guided, 75% independent. This is an **engineering challenge**: the guided part gives you a working reference path; the challenge does not.

## 1. Requirements and acceptance criteria

Public validators check these requirements. They report which requirement failed and why — never the expected implementation.

| ID | Requirement |
| --- | --- |
| RED-01 | At least 4 vulnerabilities documented |
| RED-02 | Each has asset/impact/control/test |
| RED-03 | Mutation tools least-privileged **(critical)** |
| RED-04 | Injection regression tests included **(critical)** |
| RED-05 | Resource limits defined |
| RED-06 | Zero known high-severity planted vulnerabilities unresolved **(critical)** |

**Competency gate (portal):** Zero known high-severity planted vulnerabilities unresolved. Quiz >= 80%, the guided lab, the artifact and your reflection are also required. A critical requirement cannot be offset by other scores.

## 2. Environment setup

In [ ]:
# Idempotent: safe to re-run, and it never prints secrets.
LAB_ID = "lab-13"
LABKIT_VERSION = "1.1.1"
LABKIT_SOURCES = [
    "https://agentic-ai.es/academy/fmse/labs",  # agentic-ai.es mirror
    "https://raw.githubusercontent.com/farithjhg/fmse-course-labs/main",  # GitHub (public repositories only)
]
import hashlib, json, os, pathlib, sys, urllib.request


def _fmse_bootstrap():
    """Make fmse_labkit importable: installed copy, local checkout, cached download, or verified download."""
    try:
        import fmse_labkit
        if fmse_labkit.__version__ == LABKIT_VERSION:
            return "fmse_labkit already loaded"
    except ImportError:
        pass
    here = pathlib.Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if (folder / "fmse_labkit" / "__init__.py").exists():
            sys.path.insert(0, str(folder))
            return f"using the labkit in {folder}"
    target = pathlib.Path("fmse_labkit_dist").resolve()
    marker = target / f".labkit-{LABKIT_VERSION}"
    if marker.exists():
        sys.path.insert(0, str(target))
        return "using the previously downloaded labkit"
    problems = []
    for base in [os.environ.get("FMSE_LABKIT_URL")] + LABKIT_SOURCES:
        if not base:
            continue
        base = base.rstrip("/")
        try:
            manifest = json.loads(urllib.request.urlopen(base + "/manifest.json", timeout=20).read())
            if manifest.get("labkit_version") != LABKIT_VERSION:
                raise RuntimeError(f"mirror serves labkit {manifest.get('labkit_version')}")
            for entry in manifest["files"]:
                if not entry["path"].startswith("fmse_labkit/"):
                    continue
                data = urllib.request.urlopen(f"{base}/{entry['path']}", timeout=20).read()
                if hashlib.sha256(data).hexdigest() != entry["sha256"]:
                    raise RuntimeError(f"integrity check failed for {entry['path']}")
                dest = target / entry["path"]
                dest.parent.mkdir(parents=True, exist_ok=True)
                dest.write_bytes(data)
            marker.write_text("ok")
            sys.path.insert(0, str(target))
            return f"downloaded and verified the labkit from {base}"
        except Exception as exc:  # try the next source
            problems.append(f"{base} ({type(exc).__name__})")
    raise RuntimeError(
        "Could not load fmse_labkit from: " + "; ".join(problems) + ". "
        "Recovery: download the labs folder from agentic-ai.es/academy/fmse/resources, upload the fmse_labkit folder next to this notebook (Files panel), and re-run this cell."
    )


print(_fmse_bootstrap())
import fmse_labkit as fmse

fmse.set_language("en")
print(f"fmse_labkit {fmse.__version__} ready for {LAB_ID} ({fmse.title(fmse.lab_spec(LAB_ID))})")

## 3. Secrets check

This lab runs fully offline; no provider key is needed. Keys you use elsewhere belong in Colab Secrets and are never printed.

In [ ]:
fmse.secrets_check()

## 4. Guided experiment

`SupportAgent()` with no hooks, scopes or limits is the vulnerable build. Its 'model' treats any imperative text — from the user or from a retrieved document — as an instruction. Five attacks are prepared.

In [ ]:
from fmse_labkit.labs import lab13

guided = fmse.GuidedLog(LAB_ID, required_steps=["attacks_run", "traces_inspected"])
vulnerable = lab13.vulnerable_agent()
for aid, a in lab13.ATTACKS.items():
    print(aid, f"[{a['severity']}]", "-", a["request"][:60])

## 5. Predict before you run

Which of the five attacks will succeed against the vulnerable agent? Write your prediction, then run the experiment.

In [ ]:
my_prediction = ""
guided.predict("prediction", my_prediction)

In [ ]:
results = {aid: lab13.run_attack(vulnerable, aid) for aid in lab13.ATTACKS}
for aid, res in results.items():
    print(f"{aid:<36} exploited={lab13.exploited(aid, res)}")
guided.step("attacks_run")
r = results["A1-indirect-injection-exfiltration"]
print("executed:", [(c.tool, c.args.get("to"), c.origin) for c in r.executed])
print("log:", r.log)
guided.step("traces_inspected")
guided.outcome("prediction", "see output above")

**Worksheet.** At least one full sentence per field; these observations are guided-lab evidence.

In [ ]:
worksheet = {
    "trace_observation": "",
    "boundary_observation": "",
    "prediction_check": "",
}
guided.observe(worksheet)

## 6. Challenge

**Find, fix and prove four planted vulnerabilities.** Discover at least four planted vulnerabilities, implement architectural mitigations, and prove the mitigations with regression tests.

Constraints:
- Mitigations live in the application: scopes, guards, limits, output handling, log redaction.
- Every high-severity attack must fail against your agent.

In [ ]:
def build_agent():
    """Return lab13.SupportAgent(hooks=..., tool_scopes=..., limits=...) with your architectural mitigations."""
    raise NotImplementedError


report = []  # {"id", "title", "asset", "impact", "control", "test" (a regression test id), "severity"}
regression_tests = []  # {"id", "attack": attack id, "check": fn(agent) -> True when the agent is safe}

## 7. Public validation

In [ ]:
submission = {"build_agent": build_agent, "report": report, "regression_tests": regression_tests}
result = fmse.check_public(LAB_ID, submission)
result.show()

## 8. Robustness / adversarial probes

Extra adversarial conditions beyond the graded suite. They do not change your score; they show where your solution is brittle.

In [ ]:
probe_results = fmse.probe(LAB_ID, submission)

## 9. Engineering reflection

Graded in the portal and stored with your artifact. Draft it here if useful:
- **Decision:** Which control addresses the indirect injection, and why can it not live in the prompt?
- **Trade-offs:** Which legitimate behaviour did your controls restrict, and how did you keep the agent useful?
- **Remaining risks:** Which attack class have you not tested, and what would it target?

In [ ]:
reflection_draft = {"decision": "", "tradeoffs": "", "remaining_risks": ""}

## 10. Export artifact / completion result

Exports the **Threat Model + Red-Team Report + Security Regression Suite** artifact and prints a completion record. Paste the record into the Module 13 page on agentic-ai.es (**Import lab result**). The record is a learning-workflow document, not a signed certificate.

In [ ]:
artifact = fmse.export_artifact(LAB_ID, {"report": report, "regression_tests": [{"id": t.get("id"), "attack": t.get("attack")} for t in regression_tests]},
                     title="Threat Model + Red-Team Report + Security Regression Suite", fmt="json")
record = fmse.make_completion_record(
    LAB_ID,
    result,
    {"reflection_draft": reflection_draft, **{}},
    artifact=artifact,
    guided=guided,
    probes=probe_results,
)
fmse.show_record(record)